In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf


tickers = ["JNJ", "AAPL", "MSFT", "PG", "KO"]
start = "2017-01-01"
end = "2020-12-31"

stock_data = {}

/Users/negin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Download market indext data for S&P 500 index ticker
sp500 = yf.download("^GSPC", start=start, end=end, auto_adjust=True, progress=False)
sp500.columns = sp500.columns.droplevel(1)
sp500_market_return = sp500['Close'].pct_change().to_frame(name='market_return')

In [3]:
# Download stock data for each ticker and store it in a dictionary
for ticker in tickers:
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    # Drop stock name level from columns
    df.columns = df.columns.droplevel(1)
    # Ensure datetime index and sort by date
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True)
    # Calculate stock returns variable
    df['stock_return'] = df['Close'].pct_change()
    # Target column is the stock return of the next day, so we shift the 'stock_return' column by -1
    df['target'] = df['stock_return'].shift(-1)
    # Add market indext return to the stock data
    df = df.merge(sp500_market_return, left_index=True, right_index=True, how='inner')

    # Add the stock data to the dictionary
    stock_data[ticker] = df

df = stock_data['AAPL']
df

,Close,High,Low,Open,Volume,stock_return,target,market_return
Date,,,,,,,,
2017-01-03,26.745861,26.787310,26.425786,26.665267,115127600,NaN,-0.001120,NaN
2017-01-04,26.715916,26.828749,26.653744,26.676770,84472400,-0.001120,0.005086,0.005722
2017-01-05,26.851780,26.909347,26.667563,26.692893,88774400,0.005086,0.011148,-0.000771
2017-01-06,27.151134,27.208702,26.819545,26.890928,127007600,0.011148,0.009160,0.003517
2017-01-09,27.399828,27.501147,27.158046,27.160347,134247600,0.009160,0.001008,-0.003549
...,...,...,...,...,...,...,...,...
2020-12-23,127.364159,128.793782,127.189093,128.531207,88223700,-0.006976,0.007712,0.000746
2020-12-24,128.346390,129.795483,127.500283,127.714243,54930100,0.007712,0.035766,0.003537
2020-12-28,132.936798,133.568945,129.844106,130.310937,124486200,0.035766,-0.013315,0.008723


### Preprocessing Functions

In [ ]:
def handle_outliers_winsorize(df, col, lower_q=0.01, upper_q=0.99):
    lower = df[col].quantile(lower_q)
    upper = df[col].quantile(upper_q)
    df[col] = df[col].clip(lower, upper)
    return df

### Feature Engineering Functions

In [5]:
def add_moving_average(df, n, price_col='Close'):
    """
    Calculate n-day Simple Moving Average (SMA) on a financial time series.

    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing price data
    n : int
        Window size (e.g., 10, 50, 200)
    price_col : str
        Column name to calculate SMA on (default: 'Close')

    Returns:
    --------
    df : pandas.DataFrame
        DataFrame with new SMA column added
    """
    
    df[f'SMA_{n}'] = df[price_col].rolling(window=n, min_periods=n).mean()
    return df

def add_volatility(df, n=20, price_col='Close', annualize=False):
    """
    Add an n-day rolling volatility feature based on past returns.

    This is a rolling-window feature engineering step:
    - returns describe day-to-day price changes
    - rolling std of returns estimates recent noise / uncertainty
    - if annualize=True, daily volatility is scaled by sqrt(252)

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing price data.
    n : int
        Rolling window size.
    price_col : str, default='Close'
        Column used to compute returns.
    annualize : bool, default=False
        If True, annualise daily volatility using sqrt(252).

    Returns
    -------
    pandas.DataFrame
        Copy of df with:
        - 'return'
        - f'volatility_{n}' (or annualised version)
    """

    # Rolling volatility = recent standard deviation of returns
    vol_col = f'volatility_{n}'
    df[vol_col] = df['stock_return'].rolling(window=n, min_periods=n).std()

    # Optional annualisation for finance applications
    if annualize:
        df[vol_col] = df[vol_col] * np.sqrt(252)

    return df

def add_momentum_rsi(df, n=14, price_col='Close', col_name=None):
    """
    Adds Relative Strength Index (RSI) feature to dataframe.

    Parameters:
    - df: pandas DataFrame
    - n: lookback window (default = 14, standard in finance)
    - price_col: column name for price (default = 'Close')
    - col_name: optional custom column name

    Returns:
    - df with RSI column added
    """
    
    if col_name is None:
        col_name = f'RSI_{n}'
    
    # Price changes (returns in price space, not percentage)
    delta = df[price_col].diff()

    # Gains and losses
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)

    gain = pd.Series(gain, index=df.index)
    loss = pd.Series(loss, index=df.index)

    # Rolling averages (Wilder's smoothing approximation using mean)
    avg_gain = gain.rolling(window=n, min_periods=n).mean()
    avg_loss = loss.rolling(window=n, min_periods=n).mean()

    # Avoid division by zero
    rs = avg_gain / (avg_loss + 1e-10)

    # RSI formula
    rsi = 100 - (100 / (1 + rs))

    df[col_name] = rsi

    return df

In [ ]:
# Handle initiall missing values (calendar gaps) using forward fill
df['Close'].fillna(method='ffill', inplace=True) 
# Feature Engineering: Add technical indicators as features
add_moving_average(df,10)
add_moving_average(df,50)
add_moving_average(df,200)
add_volatility(df)
add_momentum_rsi(df)

# Check for missing values after feature engineering. These can arise from rolling calculations (e.g., first 9 rows for SMA_10 will be NaN).
df.dropna(inplace=True) 

# Handle outliers
handle_outliers_winsorize(df,'stock_return')
handle_outliers_winsorize(df,'market_return')

,Close,High,Low,Open,Volume,stock_return,target,market_return,SMA_10,SMA_50,SMA_200,volatility_20,RSI_14
Date,,,,,,,,,,,,,
2017-10-17,37.411148,37.504401,37.122060,37.250284,75989200,0.003690,-0.004424,0.000673,36.504250,36.856443,33.786773,0.009765,75.913675
2017-10-18,37.245625,37.467106,37.208326,37.399494,65496800,-0.004424,-0.023661,0.000742,36.650660,36.857871,33.839272,0.008968,77.457734
2017-10-19,36.364365,36.620814,36.140557,36.543879,170336800,-0.023661,0.001731,0.000328,36.664414,36.837122,33.887515,0.009751,56.309410
2017-10-20,36.427315,36.777017,36.359707,36.511243,95896400,0.001731,-0.000512,0.005117,36.686561,36.841458,33.935392,0.009423,58.299236
2017-10-23,36.408669,36.763034,36.252469,36.576526,87937200,-0.000512,0.005955,-0.003972,36.694255,36.835350,33.981680,0.009125,55.988668
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-12-22,128.258850,130.719380,126.090071,127.996260,168904800,0.028464,-0.006976,-0.002073,122.576315,115.978038,95.552750,0.016426,69.298276
2020-12-23,127.364159,128.793782,127.189093,128.531207,88223700,-0.006976,0.007712,0.000746,123.469109,116.173881,95.856816,0.016705,67.006044
2020-12-24,128.346390,129.795483,127.500283,127.714243,54930100,0.007712,0.035766,0.003537,124.318137,116.387621,96.198654,0.016705,70.334890


In [8]:
print(df.describe())

            Close        High         Low        Open        Volume  \
count  806.000000  806.000000  806.000000  806.000000  8.060000e+02   
mean    60.739836   61.410195   59.999796   60.684911  1.337805e+08   
std     24.973674   25.385607   24.524252   24.991223  6.018671e+07   
min     33.768078   34.606402   33.722955   34.193175  4.544800e+07   
25%     41.972939   42.602259   41.793449   42.013876  9.233820e+07   
50%     49.931587   50.330934   49.512022   49.963100  1.186556e+08   
75%     71.920588   72.680324   70.529419   71.573138  1.587294e+08   
max    132.936798  134.979127  130.651317  134.259455  4.265100e+08   

       stock_return      target  market_return      SMA_10      SMA_50  \
count    806.000000  806.000000     806.000000  806.000000  806.000000   
mean       0.001758    0.001781       0.000573   60.225545   58.200670   
std        0.019668    0.021612       0.011909   24.443737   22.542023   
min       -0.063217   -0.128647      -0.043240   35.778154   36.